**Step 1 — Imports & Paths**

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from IPython.display import display
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_colwidth", 80)

CSV_PATH = Path("dataset/jamendo_with_bpm.csv")  # แก้พาธตามโครงของคุณ
DEFAULT_SONG_MINUTES = 3.5                           # ถ้าไม่มี duration_sec จะใช้ค่านี้ (นาที)
# 1) ทำซ้ำผลลัพธ์สุ่มได้ (เช่น สุ่มลิสต์เพลง)
SEED = 42
np.random.seed(SEED)

# 2) ค่าดีฟอลต์เป็นวินาที (สะดวกตอนคำนวณ)
DEFAULT_SONG_SECONDS = int(DEFAULT_SONG_MINUTES * 60)

# 3) เช็คไฟล์ให้ชัดก่อนอ่านใน Step ถัดไป
assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH.resolve()}"

# 4) (ทางเลือก) รองรับการตั้งพาธจาก ENV เวลาโยกเครื่อง/เซิร์ฟเวอร์
import os
CSV_PATH = Path(os.getenv("JAMENDO_CSV", "dataset/jamendo_with_bpm.csv"))

# 5) (ทางเลือก) โครง config รวม ใช้ส่งต่อให้ฟังก์ชันง่ายขึ้น
CONFIG = {
    "csv_path": CSV_PATH,
    "default_song_minutes": DEFAULT_SONG_MINUTES,
    "default_song_seconds": DEFAULT_SONG_SECONDS,
    "seed": SEED,
}


**Step 2 — Load & Basic Clean**

In [ ]:
VERBOSE = False  # ตั้งเป็น True ถ้าอยากเห็นสรุปและตัวอย่างแถว

df = pd.read_csv(CSV_PATH)

# ปรับชื่อคอลัมน์ให้เป็นมาตรฐาน (กันชื่อแตกต่าง)
cols_map = {
    "title": "track_title",
    "song_title": "track_title",
    "artist_name": "artist",
    "duration": "duration_sec",
}
df = df.rename(columns={k:v for k,v in cols_map.items() if k in df.columns})

needed = ["genre", "track_title", "artist", "bpm"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"CSV ขาดคอลัมน์จำเป็น: {missing}")

# แก้ bpm/duration
df["bpm"] = pd.to_numeric(df["bpm"], errors="coerce")
df = df.dropna(subset=["bpm"]).reset_index(drop=True)

if "duration_sec" in df.columns:
    df["duration_sec"] = pd.to_numeric(df["duration_sec"], errors="coerce")
else:
    df["duration_sec"] = np.nan

# ใส่ duration ถ้าหายไป
mask_dur = ~df["duration_sec"].notna()
df.loc[mask_dur, "duration_sec"] = DEFAULT_SONG_MINUTES * 60

# 0) นับแถวก่อน–หลัง เพื่อดูว่าหายไปกี่แถว
n0 = len(df)

# 1) เคลียร์ชื่อคอลัมน์/ค่า string
df.columns = df.columns.str.strip()
for c in ["genre", "track_title", "artist"]:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

# 2) กรอง BPM ให้อยู่ในช่วงสมเหตุสมผล
df = df[df["bpm"].between(60, 220)]

# 3) เติม/ซ่อม duration ที่ผิดปกติ
DEFAULT_SONG_SECONDS = int(DEFAULT_SONG_MINUTES * 60)
df.loc[
    (df["duration_sec"].isna()) | (df["duration_sec"] < 15) | (df["duration_sec"] > 15*60),
    "duration_sec"
] = DEFAULT_SONG_SECONDS

# 4) ตัดเพลงซ้ำ
dedup_keys = [c for c in ["track_title", "artist", "bpm"] if c in df.columns]
if dedup_keys:
    df = df.drop_duplicates(subset=dedup_keys, keep="first").reset_index(drop=True)

# 5) แสดงผล (ควบคุมด้วย VERBOSE)
if VERBOSE:
    print(f"Cleaned rows: {len(df)}/{n0} kept ({len(df)/max(n0,1):.1%})")
    display(df[["genre","track_title","artist","bpm","duration_sec"]].head(3))

# โชว์เฉพาะรายชื่อคอลัมน์ (เป็นค่าผลลัพธ์บรรทัดสุดท้ายของเซลล์)
df.columns.tolist()
display(df[["genre","track_title","artist","bpm","duration_sec"]].head(10))

**Step 3 — ความชอบแนวเพลง (จาก Model 1)**
- เป้าหมายของ Step นี้:

1.  รับ “คะแนนความชอบแนวเพลง” จาก Model 1 (จะมาแบบนับถี่/สัดส่วน/ความน่าจะเป็นก็ได้)

2.  แปลงให้เป็น weight 0–1 ที่รวมกันเป็น 1

3.  ทำ canonicalization ของชื่อแนวเพลง (กันสะกดต่างกัน: Hip hop/Hip-hop, K pop/K-pop ฯลฯ)

4.  ผูก weight นี้กลับเข้า df เป็นคอลัมน์ user_weight เพื่อใช้คูณกับตัวกรองโซนใน Step ถัดไป

โค้ดด้านล่าง “ปลอดภัยต่อของจริง” (กัน key ไม่ตรง, ค่าว่าง, และกรณีผู้ใช้ไม่ได้ระบุทุก genre)

In [ ]:
# ==============================
# Step 3 — User Genre Preference
# ==============================

from collections import defaultdict
import re

# >>> 3.0 ตัวอย่าง input จาก Model 1 (แทนที่ด้วยของจริงได้เลย)
#    - จะเป็นสัดส่วน, คะแนน, หรือความน่าจะเป็นก็ได้
user_pref_raw = {
    "Rock": 0.90,
    "Pop": 0.50,
    "Metal": 0.35,
    "Hip hop": 0.25,
    "EDM": 0.20,
    # แนวอื่น ๆ ที่ผู้ใช้ไม่ใส่ = 0 (เดี๋ยวเราตั้ง default ให้)
}

# >>> 3.1 ฟังก์ชัน normalize ชื่อแนวเพลงให้เป็นรูปแบบเดียวกับใน df
def norm_genre(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

# map ชื่อแปรรูป -> ชื่อมาตรฐาน (แก้ตาม taxonomy ใน df ของคุณได้)
CANON = {
    "hip-hop": "Hip hop",
    "hip hop": "Hip hop",
    "r&b": "R&B",
    "kpop": "K pop",
    "k-pop": "K pop",
    "k pop": "K pop",
    "video game music": "Video game music",
    "lofi": "Lofi",
    # เพิ่มได้ตามต้องการ...
}

def to_canon(s: str) -> str:
    ns = norm_genre(s)
    return CANON.get(ns, s if s is not None else "")

# >>> 3.2 รวมคะแนนตามชื่อมาตรฐาน (กันกรณีสะกดต่างกัน)
pref_acc = defaultdict(float)
for g, v in (user_pref_raw or {}).items():
    cg = to_canon(g)
    try:
        pref_acc[cg] += float(v)
    except Exception:
        pass

# >>> 3.3 แปลงให้ sum=1 (ถ้าไม่มีค่าเลย จะให้ Rock=1 เฉพาะกิจเพื่อไม่ให้หาร 0)
if not pref_acc:
    pref_acc["Rock"] = 1.0

total = sum(pref_acc.values()) or 1.0
user_pref = {g: (v / total) for g, v in pref_acc.items()}

# >>> 3.4 default weight สำหรับแนวที่ผู้ใช้ไม่ระบุ
#     - 0.00 = ไม่แนะนำเลย
#     - 0.02~0.05 = เปิดทาง “exploration” หน่อย ๆ
DEFAULT_WEIGHT = 0.02
pref_map = defaultdict(lambda: DEFAULT_WEIGHT, user_pref)

# >>> 3.5 ผูก weight เข้ากับ df ทีละแถว
#     - ทำคอลัมน์ genre_norm ให้ตรงกับ CANON ก่อน
df["genre_norm"] = df["genre"].map(to_canon)
df["user_weight"] = df["genre_norm"].map(lambda g: float(pref_map[g]))

# (ตัวอย่าง) ดูตารางตัวอย่าง 10 แถว (ไม่สรุปเป็น %)
from IPython.display import display
display(df[["genre", "genre_norm", "track_title", "artist", "bpm", "duration_sec", "user_weight"]].head(10))

# (ตัวเลือก) ตรวจว่าใน df ยังมีแนวที่ pref_map ไม่ครอบคลุมไหม
uncovered = sorted(set(df["genre_norm"].unique()) - set(user_pref.keys()))
# print("Uncovered genres:", uncovered)  # เปิดดูได้ถ้าต้องการ


**Step 4 — โซนความเร็วตามแคเดนซ์ (Jack Daniels-inspired)**
- เราใช้ “cadence × factor” เพื่อคำนวณ target bpm ที่เหมาะกับแต่ละโซน
โซน: E/M/T/I/R ใช้กรอบ SPM (2*spm จะ ~BPM เป้าหมาย) และเปิดรับทั้ง “1x” (ตรงจังหวะ) และ “half” (ครึ่งจังหวะ)
แนวคิดของสเต็ปนี้: แผนฝึกให้ “โซน” (E/M/T/I/R) ซึ่งสัมพันธ์กับ ความถี่ก้าว (SPM: steps per minute) ของนักวิ่ง → เราแปลงเป็น BPM ของเพลง เพื่อใช้คัดเพลงให้ จังหวะเดิน/วิ่งตรงกับร่างกาย

In [ ]:
# ==============================
# Step 4 — Cadence → Target BPM
# ==============================

# 4.1 ช่วง SPM ต่อโซน (ปรับตามงานรีวิวที่คุณใช้)
zone_spm_ranges = {
    "E": (155, 170),  # Easy
    "M": (160, 175),  # Marathon
    "T": (165, 180),  # Threshold
    "I": (170, 185),  # Interval
    "R": (175, 190),  # Repetition
}

# 4.2 factor ของโหมดการนับจังหวะ (เพลงมักถูกตรวจครึ่ง/สองเท่า)
MODE_FACTORS = {
    "double": 2.0,   # BPM ≈ 2 × SPM   (พบบ่อยจากตัวตรวจ BPM)
    "single": 1.0,   # BPM ≈ 1 × SPM
    "half":   0.5,   # BPM ≈ 0.5 × SPM (ไม่ค่อยได้ใช้ แต่กัน edge case)
}

def spm_to_targets(spm: float, modes=("double", "single")):
    """แปลง SPM เดียวเป็นรายการ BPM เป้าหมายตามโหมดที่เลือก"""
    return [MODE_FACTORS[m] * spm for m in modes if m in MODE_FACTORS]

def zone_spm_mid(zone: str) -> float:
    """คืนค่า SPM กลางของโซน"""
    lo, hi = zone_spm_ranges[zone]
    return 0.5 * (lo + hi)

# 4.3 สร้าง "หน้าต่าง BPM เป้าหมาย" ต่อโซน
def zone_target_windows(zone: str,
                        modes=("double", "single"),
                        tol_pct: float = 0.05,
                        tol_bpm: float | None = None,
                        user_spm: float | None = None):
    """
    สร้างช่วง BPM เป้าหมายสำหรับโซน
    - tol_pct: อนุโลมเป็นสัดส่วน (เช่น 0.05 = ±5%)
    - tol_bpm: ถ้าระบุ จะใช้ ±tol_bpm แทนเปอร์เซ็นต์
    - user_spm: ถ้ามี ใช้ค่านี้แทน SPM กลางของโซน (ส่วนตัวแต่ละคน)
    return: list[(lo, hi, center_bpm, mode)]
    """
    spm = float(user_spm) if user_spm is not None else zone_spm_mid(zone)
    centers = spm_to_targets(spm, modes=modes)
    windows = []
    for m, c in zip(modes, centers):
        if tol_bpm is not None:
            lo, hi = c - tol_bpm, c + tol_bpm
        else:
            lo, hi = c * (1 - tol_pct), c * (1 + tol_pct)
        windows.append((lo, hi, c, m))
    return windows

# 4.4 ตัวอย่างใช้งาน: target windows ของโซน T (Threshold)
t_windows = zone_target_windows("T", modes=("double", "single"), tol_pct=0.05)
# ตัวอย่างผล: [(~314, ~346, ~330, 'double'), (~157, ~173, ~165, 'single')]


**Step5 — ฟังก์ชันให้คะแนน BPM**
- ให้คะแนนเพลงเทียบกับเป้าหมาย BPM: Gaussian ชอบใกล้เป้า (ยิ่งใกล้=คะแนนสูง)

In [ ]:
# ใช้ MODE_FACTORS และ zone_target_windows จาก Step 4

def bpm_score_gaussian(song_bpm: float, centers: list[float], sd: float = 4.0) -> float:
    """ให้คะแนนจากระยะห่าง BPM กับ 'center' หลายค่า (เอาค่าสูงสุด)"""
    if song_bpm is None or np.isnan(song_bpm) or not centers:
        return 0.0
    sd = max(1e-6, float(sd))  # กัน sd = 0
    vals = [np.exp(-0.5 * ((song_bpm - c)/sd)**2) for c in centers]
    return float(np.max(vals))

def bpm_score_for_zone(song_bpm: float,
                       zone: str,
                       modes=("double","single"),
                       sd: float = 4.0,
                       user_spm: float | None = None) -> float:
    """คำนวณคะแนน BPM ต่อโซน ด้วย Gaussian รอบ center ของแต่ละโหมด"""
    windows = zone_target_windows(zone, modes=modes, tol_pct=0.0, user_spm=user_spm)
    centers = [c for _,_,c,_ in windows]  # ดึงค่า center_bpm
    return bpm_score_gaussian(float(song_bpm), centers, sd=sd)


**Step 6 — Blend: ความชอบ × BPM (โหมด “zone-integrate”)**
- เป้าหมายของสเต็ปนี้คือ “รวม” คะแนนจากสองฝั่ง:

    score_pref: น้ำหนักความชอบแนวเพลงจาก Step 3

    score_bpm: ความเข้าใกล้ BPM เป้าหมายของโซนจาก Step 5

    แล้วจัดอันดับเพื่อเลือกเพลงสำหรับโซน/ช่วงฝึกนั้น ๆ

    ด้านล่างคือคำอธิบายแนวคิด + เวอร์ชันโค้ดที่เก็บ edge-case และตั้งชื่อชัดขึ้น (ใช้กับของเดิมได้ทันที)***

In [ ]:
# ==============================
# Step 6 — Blend preference × BPM
# ==============================

from collections import defaultdict

def _minmax(s):
    lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo <= 1e-12:
        return pd.Series(np.zeros(len(s), dtype=float), index=s.index)
    return (s - lo) / (hi - lo)

def build_pref_map_from_pairs(pref_pairs, default_weight=0.0):
    """รับ list[(genre, weight)] -> defaultdict เพื่อ map เข้า df ได้ทนทาน"""
    base = defaultdict(float)
    for g, w in (pref_pairs or []):
        try:
            base[g] += float(w)
        except Exception:
            pass
    # normalize sum=1 ถ้ามีค่า, ไม่งั้นปล่อยเป็นศูนย์ (จะไปดึงจาก df.user_weight แทน)
    s = sum(base.values())
    if s > 0:
        for k in list(base.keys()):
            base[k] = base[k] / s
    return defaultdict(lambda: default_weight, base)

def blend_pref_bpm(
    df,
    pref_pairs=None,
    zone=None,
    user_cadence=None,      # SPM ขณะนั้น (ถ้าใช้ method='realtime')
    method="zone",          # 'zone' หรือ 'realtime'
    modes=("double","single"),
    sd=4.0,
    w_pref=0.6,
    w_bpm=0.4,
    topk=50,
    normalize_each=True,    # นอร์มคะแนนก่อน blend
    use_df_user_weight=True # ถ้าไม่มี pref_pairs ให้ใช้ df['user_weight'] จาก Step 3
):
    """
    คืน df Top-K ที่มีคอลัมน์: score_pref, score_bpm, score_final
    """
    assert abs((w_pref + w_bpm) - 1.0) < 1e-9, "w_pref + w_bpm ต้องรวมเป็น 1.0"
    out = df.copy()

    # ---------- 1) คะแนนความชอบ (score_pref) ----------
    if pref_pairs is not None and len(pref_pairs) > 0:
        pref_map = build_pref_map_from_pairs(pref_pairs, default_weight=0.0)
        out["score_pref"] = out["genre_norm"].map(lambda g: float(pref_map[g])).fillna(0.0)
    elif use_df_user_weight and "user_weight" in out.columns:
        out["score_pref"] = out["user_weight"].astype(float).fillna(0.0)
    else:
        out["score_pref"] = 0.0

    # ---------- 2) คะแนน BPM (score_bpm) ----------
    if method == "zone":
        if zone is None:
            raise ValueError("ต้องระบุ zone เมื่อ method='zone'")
        out["score_bpm"] = out["bpm"].apply(
            lambda b: bpm_score_for_zone(b, zone, modes=modes, sd=sd)
        )
    elif method == "realtime":
        if user_cadence is None:
            raise ValueError("ต้องระบุ user_cadence (SPM) เมื่อ method='realtime'")
        # ใช้ zone_target_windows ร่วมกับ user_spm เพื่อดึง centers จาก cadence จริง
        centers = [c for _,_,c,_ in zone_target_windows(zone, modes=modes, tol_pct=0.0, user_spm=float(user_cadence))]
        out["score_bpm"] = out["bpm"].apply(lambda b: bpm_score_gaussian(b, centers, sd=sd))
    else:
        raise ValueError("method ต้องเป็น 'zone' หรือ 'realtime'")

    # ---------- 3) Normalize แล้ว blend ----------
    sp = _minmax(out["score_pref"]) if normalize_each else out["score_pref"]
    sb = _minmax(out["score_bpm"]) if normalize_each else out["score_bpm"]
    out["score_final"] = (w_pref * sp) + (w_bpm * sb)

    # ---------- 4) จัดอันดับและคืน Top-K ----------
    cols_show = ["genre","genre_norm","track_title","artist","bpm","duration_sec",
                 "score_pref","score_bpm","score_final"]
    cols_show = [c for c in cols_show if c in out.columns]
    out = out.sort_values("score_final", ascending=False).reset_index(drop=True)
    return out.loc[:, cols_show].head(int(topk))


**Step 7 — DEMO: Zone-integrate (Day 1 = Interval “I”)**
- เป้าหมาย: ทดสอบ pipeline แบบครบช่วงสำหรับ 1 โซน (เช่น Interval = “I”) โดยดึง

    น้ำหนักความชอบแนวเพลงจาก Step 3 → pref_example

    สูตร Blend จาก Step 6 → คำนวณ score_pref, score_bpm, score_final
    แล้วดู Top-N เพลงที่เหมาะสุด

In [ ]:
# ==============================
# Step 7 — DEMO: Zone-integrate
# ==============================

# 1) ใช้น้ำหนักจาก Step 3
pref_example = list(user_pref.items())  # [(genre, weight_norm), ...]

# 2) เลือกโซนทดสอบ (ตัวอย่าง: Interval)
zone = "I"

# 3) เรียก blend (ใช้โหมดชื่อเดียวกับ Step 4–5)
ranked = blend_pref_bpm(
    df,
    pref_pairs=pref_example,
    zone=zone,
    method="zone",                 # หรือ "realtime" + user_cadence=172
    modes=("double","single"),
    sd=4.0,
    w_pref=0.6,
    w_bpm=0.4,
    topk=50
)

# 4) แสดงผลดู 10 อันดับแรก
cols_show = ["genre","track_title","artist","bpm","score_pref","score_bpm","score_final"]
ranked[cols_show].head(10)


**Step 8 — DEMO: Realtime cadence (ถ้ามี SPM จากนาฬิกา)**
- เป้าหมายของสเต็ปนี้คือ “ลองจับคู่แบบเวลาจริง” โดยใช้ user_cadence_now (SPM ปัจจุบัน) แทน SPM กลางโซน แล้วจัดอันดับเพลงเหมือน Step 7

In [ ]:
# ==============================
# Step 8 — DEMO: Realtime cadence
# ==============================

user_cadence_now = 176  # SPM จากนาฬิกา/เซนเซอร์

ranked_rt = blend_pref_bpm(
    df,
    pref_pairs=list(user_pref.items()),   # จาก Step 3
    zone="I",                             # โซน ณ ช่วงนี้ (ตั้งให้สอดคล้องแผน)
    method="realtime",
    user_cadence=user_cadence_now,
    modes=("double","single"),
    sd=4.0,
    w_pref=0.6, w_bpm=0.4,
    topk=50
)

ranked_rt[["genre","track_title","artist","bpm","score_pref","score_bpm","score_final"]].head(10)


**Step 9 — สร้าง “เพลย์ลิสต์แบบผสมต่อแนว” สำหรับ 1 ช่วงฝึก**
- เป้าหมาย: รับ df_ranked (ผลจาก Step 6–8 ที่เรียงตาม score_final แล้ว) + ความชอบแนวเพลง → เลือกเพลง กระจายหลายแนว และ ไม่ซ้ำศิลปิน ง่าย ๆ

In [ ]:
from collections import defaultdict, deque

def make_playlist_by_genre(
    df_ranked,
    pref_pairs,
    n_per_genre=5,
    unique_artist=True,
    artist_cap=1,                     # จำกัดเพลงต่อศิลปิน (1 = ไม่ซ้ำ)
    min_dur=60, max_dur=12*60,        # กรองความยาว (วินาที)
    use_genre_norm=True,
    topk_fallback=50,
    round_robin=True,                 # สลับคิวตามแนวเวลาเรียงจริง
):
    # ----- เตรียมฟิลด์ -----
    gcol = "genre_norm" if (use_genre_norm and "genre_norm" in df_ranked.columns) else "genre"
    cols_need = {gcol, "artist", "track_title", "bpm", "duration_sec", "score_final"}
    miss = [c for c in cols_need if c not in df_ranked.columns]
    if miss:
        raise ValueError(f"df_ranked ขาดคอลัมน์: {miss}")

    # กรองเบื้องต้น
    base = df_ranked.copy()
    base = base.drop_duplicates(subset=["track_title","artist","bpm"], keep="first")
    base = base[base["duration_sec"].between(min_dur, max_dur)]

    # แนวที่ผู้ใช้ชอบ (>0)
    liked = [g for g, w in (pref_pairs or []) if (isinstance(w,(int,float)) and w > 0)]
    if not liked:
        liked = list(base[gcol].dropna().unique())

    # จำกัดศิลปิน
    artist_count = defaultdict(int)

    # เตรียมถัง per-genre
    per_genre_bags = {g: [] for g in liked}
    for g in liked:
        sub = base[base[gcol] == g]
        # เดินตามอันดับ เพื่อเลือก n_per_genre (ไม่เกิน)
        picked = []
        for _, r in sub.iterrows():
            a = r["artist"]
            if unique_artist and artist_count[a] >= artist_cap:
                continue
            picked.append(r)
            artist_count[a] += 1
            if len(picked) >= n_per_genre:
                break
        per_genre_bags[g] = picked

    # รวมจำนวนที่เก็บได้
    bag = [r for lst in per_genre_bags.values() for r in lst]

    # เติมให้ถึงโควตรวม (ถ้าขาด) ด้วยกองรวมที่เหลือ (ยังคงเงื่อนไขศิลปิน)
    need_total = n_per_genre * max(1, len(liked))
    if len(bag) < need_total:
        already = {(r["track_title"], r["artist"], r["bpm"]) for r in bag}
        for _, r in base.iterrows():
            key = (r["track_title"], r["artist"], r["bpm"])
            if key in already:
                continue
            a = r["artist"]
            if unique_artist and artist_count[a] >= artist_cap:
                continue
            bag.append(r)
            artist_count[a] += 1
            if len(bag) >= min(need_total, topk_fallback):
                break

    if not bag:
        # fallback สุดท้าย
        return base.head(topk_fallback).reset_index(drop=True)

    out_df = pd.DataFrame(bag)

    # ----- จัดลำดับแทร็ก: round-robin ตามแนว (ช่วยให้ไม่ติดแนวเดียว) -----
    if round_robin:
        q = {g: deque(per_genre_bags[g]) for g in liked}
        rr = []
        # วนหยิบจากแต่ละแนวทีละ 1 จนหมด
        while any(q[g] for g in liked):
            for g in liked:
                if q[g]:
                    rr.append(q[g].popleft())
        out_df = pd.DataFrame(rr)

    # รูปแบบผลลัพธ์
    cols_show = ["genre","track_title","artist","bpm","duration_sec","score_final"]
    cols_show = [c for c in cols_show if c in out_df.columns]
    out_df = out_df.loc[:, cols_show].reset_index(drop=True)
    return out_df


**Step 10 — (อ็อปชัน) เตรียมคอลัมน์ “คะแนน BPM ต่อโซน” ไว้ใช้วิเคราะห์/คัดกรอง**
- แนวคิด: คำนวณ bpm_score ของทุกเพลงสำหรับทุกโซน (E/M/T/I/R) ล่วงหน้า → จะเห็น distribution ว่าเพลงชุดเราถนัดโซนไหน และใช้คอลัมน์พวกนี้ทำ “ตัวกรองผู้สมัคร” ตอนแพ็กเพลย์ลิสต์ตามเวลาช่วงฝึก

In [ ]:
# ==============================
# Step 10 — Precompute bpm_score per zone
# ==============================

ZONES = list(zone_spm_ranges.keys())  # ['E','M','T','I','R']

for z in ZONES:
    col = f"score_bpm_{z}"
    cols = [c for c in df.columns if c.startswith("score_bpm_")]
thr = 0.35  # เกณฑ์ความเข้ากัน (ปรับได้)
strong_counts = pd.Series({c: int((df[c] >= thr).sum()) for c in cols}, name=f"count>= {thr}")
strong_counts.to_frame()



In [ ]:
# =========================
# Step 11 — pick_block_songs (improved: clip-mode + remain-aware)
# =========================
from collections import namedtuple, defaultdict
import pandas as pd

Segment = namedtuple("Segment", ["zone", "duration_sec", "repeat"])

def pick_block_songs(
    df, zone, target_seconds, user_pref_pairs,
    *,
    method="zone",
    user_cadence=None,
    modes=("double","single"),
    sd=4.0,
    w_pref=0.6, w_bpm=0.4,
    tolerance=15,                 # ยอมคลาดเวลา ± วินาทีต่อบล็อก
    unique_track=True,            # กันซ้ำเพลงทั้งวัน
    artist_cap_day=2,             # โควตาศิลปิน/วัน
    artist_min_gap=3,             # ต้องเว้นห่าง >= N แทร็กก่อนศิลปินเดิมจะกลับมา
    day_seen_tracks=None,
    day_artist_count=None,
    day_artist_lastpos=None,
    use_candidates=True,          # กรองด้วย score_bpm_{zone} ถ้ามี/คำนวณสด
    thr_candidate=0.35,           # เกณฑ์ผู้สมัครขั้นต่ำ
    # --- ใหม่ ---
    clip_mode=True,               # เล่นเป็นคลิป เพื่อให้เข้าเวลา target ของบล็อก
    clip_max=70                   # ความยาวคลิปสูงสุดต่อ 1 เพลง (วินาที)
):
    if day_seen_tracks is None:  day_seen_tracks  = set()
    if day_artist_count is None: day_artist_count = defaultdict(int)
    if day_artist_lastpos is None: day_artist_lastpos = {}

    # 1) จัดอันดับเพลงสำหรับโซนนี้
    ranked = blend_pref_bpm(
        df,
        pref_pairs=user_pref_pairs,
        zone=zone,
        method=method,
        user_cadence=user_cadence,
        modes=modes, sd=sd,
        w_pref=w_pref, w_bpm=w_bpm,
        topk=200,
        normalize_each=True
    )

    # 2) Candidate filter — ถ้าไม่มีคอลัมน์ score_bpm_{zone} ให้คำนวณสด
    if use_candidates:
        ccol = f"score_bpm_{zone}"
        if ccol not in ranked.columns:
            ranked[ccol] = ranked["bpm"].apply(
                lambda b: bpm_score_for_zone(float(b), zone, modes=modes, sd=sd)
            )
        ranked = ranked[ranked[ccol] >= thr_candidate].reset_index(drop=True)

    bag, total_use, pos = [], 0, 0  # total_use = เวลาที่จะเล่นจริง (play_sec)

    # 3) เลือกแบบ greedy + penalty + ใกล้ remain
    while total_use < target_seconds - tolerance and not ranked.empty:
        cand = ranked.copy()

        # กันซ้ำเพลงทั้งวัน
        if unique_track:
            cand = cand[~cand.apply(
                lambda r: (r["track_title"], r["artist"], r["bpm"]) in day_seen_tracks,
                axis=1
            )]

        # กันเกินโควตาศิลปินทั้งวัน
        cand = cand[cand["artist"].map(lambda a: day_artist_count[a] < artist_cap_day)]
        if cand.empty:
            break

        # penalty หากศิลปินเดิมยังเว้นไม่พอ
        def penalize(row):
            base = float(row["score_final"])
            a    = row["artist"]
            last = day_artist_lastpos.get(a, -10**9)
            gap  = pos - last
            if gap < artist_min_gap:
                base *= 0.5
            return base

        # จัดตามคะแนนหลัง penalty + ใกล้เวลา remain
        remain = max(1, target_seconds - total_use)
        cand = cand.assign(
            score_penalized = cand.apply(penalize, axis=1),
            dur = cand.get("duration_sec", 180).fillna(180).astype(int)
        )
        cand["gap_to_remain"] = (cand["dur"] - remain).abs()
        cand = cand.sort_values(
            by=["score_penalized", "gap_to_remain"],
            ascending=[False, True]
        )

        r   = cand.iloc[0].copy()
        dur = int(r.get("duration_sec", 180))

        # เล่นทั้งเพลง หรือเล่นเป็นคลิปให้เข้าเวลาเป้า
        if clip_mode:
            play_sec = min(remain, clip_max, dur)
        else:
            play_sec = dur

        r["play_sec"] = int(play_sec)
        bag.append(r)

        # อัปเดตสถานะทั้งวัน
        total_use += int(play_sec)
        key = (r["track_title"], r["artist"], r["bpm"])
        day_seen_tracks.add(key)
        day_artist_count[r["artist"]] += 1
        day_artist_lastpos[r["artist"]] = pos
        pos += 1

        # ตัดผู้ที่เลือกออกจาก ranked
        ranked = ranked.drop(index=cand.index[0]).reset_index(drop=True)

        if total_use >= target_seconds + tolerance:
            break

    out_df = pd.DataFrame(bag)
    if "play_sec" not in out_df.columns:
        out_df["play_sec"] = 0  # เผื่อไม่มีเพลงเลย

    return out_df, int(total_use), day_seen_tracks, day_artist_count, day_artist_lastpos


In [ ]:
# =========================================
# White Plan Day 1 playlist — robust (fixed play_sec)
# =========================================
from collections import namedtuple, defaultdict
import pandas as pd
import numpy as np
import random

# ---- seed ให้ผลเสถียร ----
random.seed(42)
np.random.seed(42)

# -------- Config --------
BLOCK_TOL = 15          # ยอมคลาดเวลาในแต่ละบล็อก ±วินาที
W_THR     = 0.30        # เกณฑ์ผู้สมัครขั้นต่ำช่วง "เดิน" (W)
RUN_THR   = 0.35        # เกณฑ์ผู้สมัครขั้นต่ำช่วงวิ่ง
ART_CAP   = 2           # ศิลปินเดิมต่อวันได้ไม่เกินกี่เพลง
ART_GAP   = 3           # ต้องเว้นห่างกี่แทร็กก่อนศิลปินเดิมจะกลับมา
SD_GAUSS  = 4.0         # sd สำหรับ Gaussian BPM score
W_PREF    = 0.60        # น้ำหนัก score_pref
W_BPM     = 0.40        # น้ำหนัก score_bpm

# ถ้าพร้อมใช้โซน W จริง เปลี่ยนเป็น False
MAP_W_TO_E = True

Segment = namedtuple("Segment", ["zone", "duration_sec", "repeat"])

# ---------- utility: safe int ----------
def _as_int_ser(s, fill=0):
    if s is None:
        return None
    return pd.to_numeric(s, errors="coerce").fillna(fill).astype(int)

# ---------- Step 11 — pick_block_songs (fixed) ----------
def pick_block_songs(
    df, zone, target_seconds, user_pref_pairs,
    *,
    method="zone",
    user_cadence=None,
    modes=("double","single"),
    sd=4.0,
    w_pref=0.6, w_bpm=0.4,
    tolerance=15,
    unique_track=True,
    artist_cap_day=2,
    artist_min_gap=3,
    day_seen_tracks=None,
    day_artist_count=None,
    day_artist_lastpos=None,
    use_candidates=True,
    thr_candidate=0.35,
    clip_mode=True,
    clip_max=70
):
    """
    - จัดอันดับด้วย blend_pref_bpm (ต้องนิยามไว้ก่อนหน้า)
    - กรองด้วย score_bpm_{zone} (ถ้าไม่มีจะคำนวณสด)
    - remain-aware + penalty ศิลปิน
    - play_sec = min(duration, remain + tolerance) ทุกบล็อก
      และถ้าเป็นบล็อกสั้น (<=75s) ใช้ clip_max ควบคู่
    """
    if day_seen_tracks is None: day_seen_tracks = set()
    if day_artist_count is None: day_artist_count = defaultdict(int)
    if day_artist_lastpos is None: day_artist_lastpos = {}

    # 1) จัดอันดับ
    base_rank = blend_pref_bpm(
        df,
        pref_pairs=user_pref_pairs,
        zone=zone,
        method=method,
        user_cadence=user_cadence,
        modes=modes, sd=sd,
        w_pref=w_pref, w_bpm=w_bpm,
        topk=200,
        normalize_each=True
    ).reset_index(drop=True)
    ranked = base_rank.copy()

    # 2) Candidate filter (+ fallback)
    ccol = f"score_bpm_{zone}"
    if use_candidates:
        if ccol not in ranked.columns:
            ranked[ccol] = ranked["bpm"].apply(
                lambda b: bpm_score_for_zone(float(b), zone, modes=modes, sd=sd)
            )
        ranked = ranked[ranked[ccol] >= thr_candidate].reset_index(drop=True)
        if ranked.empty:
            looser = max(thr_candidate - 0.05, 0.0)
            ranked = base_rank.copy()
            if ccol not in ranked.columns:
                ranked[ccol] = ranked["bpm"].apply(
                    lambda b: bpm_score_for_zone(float(b), zone, modes=modes, sd=sd)
                )
            ranked = ranked[ranked[ccol] >= looser].reset_index(drop=True)
            if ranked.empty:
                ranked = base_rank.head(50).copy()

    # เตรียม duration_sec
    if "duration_sec" in ranked.columns:
        ranked["duration_sec"] = _as_int_ser(ranked["duration_sec"], fill=180)
    else:
        ranked["duration_sec"] = 180

    bag, total, pos = [], 0, 0

    while total < (target_seconds - tolerance) and not ranked.empty:
        cand = ranked.copy()

        # กันซ้ำเพลงทั้งวัน
        if unique_track:
            cand = cand[~cand.apply(
                lambda r: (r["track_title"], r["artist"], r["bpm"]) in day_seen_tracks,
                axis=1
            )]

        # กันเกินโควตาศิลปินทั้งวัน
        cand = cand[cand["artist"].map(lambda a: day_artist_count[a] < artist_cap_day)]
        if cand.empty:
            break

        remain = max(1, target_seconds - total)

        def penalize(row):
            base = float(row["score_final"])
            a    = row["artist"]
            last = day_artist_lastpos.get(a, -10**9)
            gap  = pos - last
            if gap < artist_min_gap:
                base *= 0.5
            return base

        cand = cand.assign(
            score_penalized=cand.apply(penalize, axis=1),
            dur=_as_int_ser(cand.get("duration_sec", 180), fill=180),
            gap_to_remain=(cand.get("duration_sec", 180).fillna(180).astype(int) - remain).abs()
        ).sort_values(
            ["score_penalized", "gap_to_remain", "bpm", "track_title"],
            ascending=[False, True, True, True]
        )

        r   = cand.iloc[0]
        dur = int(r.get("duration_sec", 180))

        # ---- FIX: play_sec ไม่เกินเวลาที่เหลือ + tolerance ทุกบล็อก ----
        play_sec = min(dur, int(remain + tolerance))
        if clip_mode and target_seconds <= 75:
            play_sec = min(play_sec, int(clip_max))

        rr = r.to_dict()
        rr["play_sec"] = int(play_sec)
        bag.append(rr)

        total += int(play_sec)

        # อัปเดตสถานะทั้งวัน
        key = (r["track_title"], r["artist"], r["bpm"])
        day_seen_tracks.add(key)
        day_artist_count[r["artist"]] += 1
        day_artist_lastpos[r["artist"]] = pos
        pos += 1

        ranked = ranked.drop(index=cand.index[0]).reset_index(drop=True)

        if total >= (target_seconds + tolerance):
            break

    return pd.DataFrame(bag), int(total), day_seen_tracks, day_artist_count, day_artist_lastpos

# ---------- กำหนด “แผน 1 วัน” White Plan ----------
# 5' W + 10 × (1' E + 1' W) + 5' W
plan_day = [Segment("W", 5*60, 1)]
for _ in range(10):
    plan_day += [Segment("E", 60, 1), Segment("W", 60, 1)]
plan_day += [Segment("W", 5*60, 1)]

# ---------- Run 1 วัน ----------
day_seen_tracks  = set()
day_artist_count = defaultdict(int)
day_artist_lastpos = {}

rows = []
block_meta = []

for seg in plan_day:
    # เลือกกลยุทธ์โซนเพลงที่จะใช้คัด
    if seg.zone == "W" and MAP_W_TO_E:
        z_used    = "E"               # map W → E พร้อมผ่อนเกณฑ์
        thr_use   = W_THR
        modes_use = ("single",)       # ช่วงเดินใช้ single
    else:
        z_used    = seg.zone
        thr_use   = RUN_THR if seg.zone != "W" else W_THR
        modes_use = ("double","single") if seg.zone != "W" else ("single",)

    block_df, tot, day_seen_tracks, day_artist_count, day_artist_lastpos = pick_block_songs(
        df, z_used, seg.duration_sec,
        user_pref_pairs=list(user_pref.items()),
        method="zone",
        modes=modes_use, sd=SD_GAUSS,
        w_pref=W_PREF, w_bpm=W_BPM,
        tolerance=BLOCK_TOL,
        artist_cap_day=ART_CAP,
        artist_min_gap=ART_GAP,
        day_seen_tracks=day_seen_tracks,
        day_artist_count=day_artist_count,
        day_artist_lastpos=day_artist_lastpos,
        use_candidates=True,
        thr_candidate=thr_use,
        clip_mode=True,
        clip_max=70
    )

    block_id = len(block_meta)
    bdf = block_df.copy()
    bdf["plan_block"]      = block_id
    bdf["plan_zone"]       = seg.zone           # โซนตามแผนการฝึก
    bdf["used_zone"]       = z_used             # โซนที่ใช้คัดเพลงจริง
    bdf["plan_target_sec"] = int(seg.duration_sec)
    rows.append(bdf)

    block_meta.append({
        "block": block_id,
        "zone": seg.zone,
        "used_zone": z_used,
        "target_sec": int(seg.duration_sec),
        "n_tracks": len(block_df),
    })

playlist_day = pd.concat(rows, ignore_index=True)

# ---------- ทำให้คอลัมน์เวลาเป็น int และเลือก use_sec ----------
if "play_sec" in playlist_day.columns:
    playlist_day["play_sec"] = _as_int_ser(playlist_day["play_sec"], fill=0)
if "duration_sec" in playlist_day.columns:
    playlist_day["duration_sec"] = _as_int_ser(playlist_day["duration_sec"], fill=0)

playlist_day["use_sec"] = playlist_day["play_sec"].where(
    playlist_day.get("play_sec", 0) > 0,
    playlist_day.get("duration_sec", 0)
).astype(int)

# ---------- block_summary ----------
meta_df = pd.DataFrame(block_meta)
block_summary = (
    playlist_day.groupby("plan_block", as_index=False)["use_sec"].sum()
    .rename(columns={"use_sec":"actual_sec"})
    .merge(meta_df.rename(columns={"block":"plan_block"}), on="plan_block", how="right")
    .sort_values("plan_block")
    .reset_index(drop=True)
)
block_summary["target_sec"] = _as_int_ser(block_summary["target_sec"], fill=0)
block_summary["actual_sec"] = _as_int_ser(block_summary["actual_sec"], fill=0)
block_summary["diff_sec"]   = block_summary["actual_sec"] - block_summary["target_sec"]
block_summary["ok"]         = (block_summary["diff_sec"].abs() <= BLOCK_TOL)
block_summary["start_sec"]  = block_summary["actual_sec"].cumsum().shift(fill_value=0)
block_summary["end_sec"]    = block_summary["start_sec"] + block_summary["actual_sec"]

# ---------- ตรวจ pattern ----------
expected  = [("W", 5*60)] + [("E", 60), ("W", 60)]*10 + [("W", 5*60)]
exp_zones = [z for z,_ in expected]
got_zones = list(block_summary["zone"])
print("แพทเทิร์นโซนถูกต้อง:", got_zones == exp_zones)

# ---------- รายงานสรุป ----------
print("\n=== Block summary ===")
display(block_summary)

print("\nรวมเวลาจริงทั้งวัน (นาที):", round(block_summary["actual_sec"].sum()/60, 1))
print("รวมเป้าหมายทั้งวัน (นาที):", round(block_summary["target_sec"].sum()/60, 1))
print("บล็อกเข้าเป้า (±{}s): {}/{}".format(BLOCK_TOL, int(block_summary["ok"].sum()), len(block_summary)))
if "artist" in playlist_day.columns:
    print("ความหลากหลายศิลปินทั้งวัน:", playlist_day["artist"].nunique(), "จาก", len(playlist_day), "แทร็ก")

# schema ที่ส่งให้เว็บแนะนำ
cols = [
    "plan_block","plan_zone","used_zone",
    "track_title","artist","bpm",
    "duration_sec","play_sec","use_sec",
    "plan_target_sec"
]
cols = [c for c in cols if c in playlist_day.columns]
print("\n=== preview เพลง (20 รายการ) ===")
display(playlist_day[cols].head(20))

# (ทางเลือก) export
# playlist_day.to_csv("day1_playlist.csv", index=False)
